In [0]:
# ============================================================
# PAYCE — Gold Layer Configuration
# Notebook: 03_gold_credit_risk_mart
# ============================================================

CATALOG = "payce"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

print("Config Loaded Successfully")

In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id, when 

In [0]:
# ============================================================
# CREDIT RISK MART — dim_grade
# LendingClub risk grades A–G, with a risk_level label
# ============================================================

from pyspark.sql.functions import col, monotonically_increasing_id, when 

In [0]:
# Distinct grades from LendingClub
dim_grade = (
    spark.table(f"{SILVER}.lending_club")
    .select("grade")
    .distinct()
    .filter(col("grade").isNotNull())
    .withColumn("grade_id", monotonically_increasing_id())

    # Add a business friendly risk level
    .withColumn(
        "risk_level",
        when(col("grade").isin("A","B"), "Low Risk")
        .when(col("grade").isin("C","D"), "Medium Risk")
        .otherwise("High Risk")
    )
    .select("grade_id", "grade", "risk_level")
)


In [0]:
# Write dim_grade to gold
dim_grade.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.dim_grade")

In [0]:
# SHOW THE TABLE
spark.table(f"{GOLD}.dim_grade").orderBy("grade_id").show()

In [0]:
# ============================================================
# CREDIT RISK MART — dim_purpose
# Distinct loan purposes from LendingClub
# ============================================================

from pyspark.sql.functions import col, monotonically_increasing_id, when

dim_purpose = (
    spark.table(f"{SILVER}.lending_club")
    .select("purpose")
    .distinct()
    .filter(col("purpose").isNotNull())
    .withColumn("purpose_id", monotonically_increasing_id())
    .select("purpose_id", "purpose")
)

In [0]:
# Write dim_purpose to Gold
dim_purpose.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.dim_purpose")


In [0]:
# SHOW THE TABLE
spark.table(f"{GOLD}.dim_purpose").orderBy("purpose_id").show()

In [0]:
# Check the most frequent purpose values
spark.table(f"{SILVER}.lending_club") \
    .groupBy("purpose") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(20, truncate=False)

In [0]:
# CLEANED VERSION OF dim_purpose

# The official valid loan purposes
valid_purposes = [
    "debt_consolidation", "credit_card", "home_improvement", "other",
    "major_purchase", "medical", "small_business", "car",
    "vacation", "moving", "house", "wedding",
    "renewable_energy", "educational"
]

dim_purpose = (
    spark.table(f"{SILVER}.lending_club")
    .select("purpose")
    .distinct()
    .filter(col("purpose").isin(valid_purposes))   # keep only real ones
    .withColumn("purpose_id", monotonically_increasing_id())
    .select("purpose_id", "purpose")
)

# Overwrite the bad version
dim_purpose.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.dim_purpose")

# Confirm — should be ~14 clean rows
spark.table(f"{GOLD}.dim_purpose").orderBy("purpose_id").show(truncate=False)

In [0]:
# ============================================================
# CREDIT RISK MART — dim_term
# Loan Terms: 36 and 60 months
# ============================================================

spark.table(f"{SILVER}.lending_club").select("term").distinct().show(truncate=False)

In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id, when, regexp_extract, trim

In [0]:
dim_term = (
    spark.table(f"{SILVER}.lending_club")
    .select("term")
    .distinct()
    .filter(col("term").isNotNull())
    # Extract just the number (36/60) from "36 months"
    .withColumn("term_months", regexp_extract(col("term"), r"(\d+)", 1).cast("int"))
    .withColumn("term_id", monotonically_increasing_id())
    .select("term_id","term_months")
)

In [0]:
# Write dim_term to gold
dim_term.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.dim_term")

In [0]:
# SHOW TABLE
spark.table(f"{GOLD}.dim_term").orderBy("term_id").show()

In [0]:
# ============================================================
# CREDIT RISK MART — fct_loans
# Grain: one row per loan
# Links to dim_grade, dim_purpose, dim_term
# ============================================================

from pyspark.sql.functions import col, regexp_extract

# Load Sources
lc = spark.table(f"{SILVER}.lending_club")
dim_grade = spark.table(f"{GOLD}.dim_grade")
dim_purpose = spark.table(f"{GOLD}.dim_purpose")
dim_term = spark.table(f"{GOLD}.dim_term")

In [0]:
# Prepare a clean numeric term on loan side to join on
lc = lc.withColumn("term_months", regexp_extract(col("term"), r"(\d+)", 1).cast("int"))
                   
# Build the fact table
fct_loans = (
    lc.alias("l")
    .join(dim_grade.alias("g"), col("l.grade") == col("g.grade"), "left")
    .join(dim_purpose.alias("p"), col("l.purpose") == col("p.purpose"), "left")
    .join(dim_term.alias("t"), col("l.term_months") == col("t.term_months"), "left")
    .select(
        col("l.id").alias("loan_id"),
        col("g.grade_id"),
        col("p.purpose_id"),
        col("t.term_id"),
        col("l.loan_amnt").alias("loan_amount"),
        col("l.int_rate"),
        col("l.dti"),
        col("l.annual_inc"),
        col("l.loan_outcome")
    )
)

In [0]:
# Write to Gold
fct_loans.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.fct_loans")

In [0]:
# Confirm
spark.table(f"{GOLD}.fct_loans").show(5)

In [0]:
%sql
SELECT * FROM payce.gold.fct_loans LIMIT 5